# Qwen3 Forced Aligner Plugin

> Plugin implementation for Qwen3 word-level forced alignment

In [ ]:
#| default_exp plugin

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Union, ClassVar

import torch
from fastcore.basics import patch

try:
    from qwen_asr import Qwen3ForcedAligner
    QWEN3_AVAILABLE = True
except ImportError:
    QWEN3_AVAILABLE = False

# Stage 8 (Option C / PILLAR 1c): the tool re-bases onto ToolCapability (pure
# compute). The cache/persist bookends moved OUT to the generic forced-alignment
# adapter (cjm-forced-alignment-adapter-interface); the ForcedAlignResult data
# noun lives in cjm-capability-primitives; identity derives from the installed
# distribution. No get_plugin_metadata, no self.storage. Audio arrives
# MODEL-READY (converted upstream by ffmpeg); text is the transcript to align.
from cjm_plugin_system.core.capability import ToolCapability, RELOAD_TRIGGER, EnvVarSpec
from cjm_capability_primitives.forced_alignment import ForcedAlignItem, ForcedAlignResult
from cjm_plugin_system.core.errors import PluginInputError
from cjm_plugin_system.utils.validation import (
    dict_to_config, config_to_dict, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_ENUM
)

# Shared plugin utils: HF Hub download/cache mixin + load + torch release + CUDA-OOM typing.
from cjm_hf_plugin_utils.cache_config import HFCacheConfig
from cjm_hf_plugin_utils.download import snapshot_download_with_progress
from cjm_hf_plugin_utils.loading import load_pretrained_with_oom
from cjm_torch_plugin_utils.memory import release_model
from cjm_torch_plugin_utils.oom import cuda_oom_to_plugin_resource_error

## Configuration

In [ ]:
#| export
@dataclass
class Qwen3ForcedAlignerConfig(HFCacheConfig):
    """Configuration for the Qwen3 Forced Aligner plugin.

    Composes HFCacheConfig (cache_dir / revision / local_files_only / trust_remote_code,
    each RELOAD_TRIGGER-tagged) so HF Hub download behaviour is operator-controllable."""

    model_id: str = field(
        default="Qwen/Qwen3-ForcedAligner-0.6B",
        metadata={
            SCHEMA_TITLE: "Model ID",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "HuggingFace model identifier",
        }
    )

    device: str = field(
        default="auto",
        metadata={
            SCHEMA_TITLE: "Device",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Device for model inference ('auto', 'cuda:0', 'cpu')",
        }
    )

    dtype: str = field(
        default="bfloat16",
        metadata={
            SCHEMA_TITLE: "Data Type",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Model precision",
            SCHEMA_ENUM: ["bfloat16", "float16", "float32"],
        }
    )

    attn_implementation: str = field(
        default="sdpa",
        metadata={
            SCHEMA_TITLE: "Attention Implementation",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Attention backend ('sdpa' for PyTorch native, 'flash_attention_2' if flash-attn installed)",
            SCHEMA_ENUM: ["sdpa", "flash_attention_2", "eager"],
        }
    )

    language: str = field(
        default="English",
        metadata={
            SCHEMA_TITLE: "Language",
            SCHEMA_DESC: "Language for alignment",
        }
    )

## Plugin Implementation

In [ ]:
#| export
class Qwen3ForcedAlignerPlugin(ToolCapability):
    """Qwen3 word-level forced-alignment tool capability (stage 8: pure compute).

    Native-surface model (PILLAR 1c): this tool is PURE COMPUTE — `align` reads
    MODEL-READY audio + the transcript text, runs Qwen3 inference, and builds the
    typed `ForcedAlignResult`. The cache-check + persistence bookends + the
    per-call `force` control live in the generic forced-alignment adapter
    (cjm-forced-alignment-adapter-interface); the result DTO lives in
    cjm-capability-primitives; identity derives from the installed distribution.
    No `get_plugin_metadata`, no `self.storage`."""

    config_class = Qwen3ForcedAlignerConfig

    # Track 19 (CR-12 worker-env model): worker spawn env declared on the class.
    # CUDA_VISIBLE_DEVICES + OMP_NUM_THREADS are static; HF_HOME is templated to the
    # substrate models dir (HF Hub model). The substrate resolves + injects at Popen.
    WORKER_ENV: ClassVar[List[EnvVarSpec]] = [
        EnvVarSpec(
            name="CUDA_VISIBLE_DEVICES",
            default="0",
            label="GPU Device",
            description="Which GPU index the worker uses.",
        ),
        EnvVarSpec(
            name="OMP_NUM_THREADS",
            default="4",
            label="OpenMP Threads",
            description="Thread cap for CPU-side ops.",
        ),
        EnvVarSpec(
            name="HF_HOME",
            default="${CJM_MODELS_DIR}/huggingface",
            label="HF Cache Directory",
            description="HuggingFace Hub cache root (templated to the substrate models dir).",
        ),
    ]

    def __init__(self):
        """Initialize the plugin with default state."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Qwen3ForcedAlignerConfig = None
        self._model = None
        self._loaded_model_id: Optional[str] = None
        self._loaded_device: Optional[str] = None
        self._loaded_dtype: Optional[str] = None
        self._loaded_attn: Optional[str] = None

    @property
    def name(self) -> str:  # Plugin name identifier
        """Plugin identity, derived from the installed distribution (PILLAR 1c)."""
        from importlib.metadata import metadata, packages_distributions
        dist = (packages_distributions().get(__package__) or [__package__.replace("_", "-")])[0]
        return metadata(dist)["Name"]

    @property
    def version(self) -> str:  # Plugin version string
        from cjm_transcription_plugin_qwen3_forced_aligner import __version__
        return __version__

    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for configuration
        """Return JSON Schema for UI generation."""
        return dataclass_to_jsonschema(Qwen3ForcedAlignerConfig)

    def get_current_config(self) -> Dict[str, Any]:  # Current configuration as dictionary
        """Return current configuration state."""
        return config_to_dict(self.config) if self.config else {}

    def initialize(
        self,
        config: Optional[Any] = None  # Configuration dataclass, dict, or None
    ) -> None:
        """First-time setup. CR-4: config application is factored into _apply_config;
        the substrate's reconfigure fires _release_model on a model_id/device/dtype/
        attn_implementation change (RELOAD_TRIGGER) then re-applies config. No storage
        init — the adapter owns the cache (stage 8)."""
        self._apply_config(config)
        self.logger.info(f"Initialized with model={self.config.model_id}, device={self.config.device}")

    def align(
        self,
        audio: Union[str, Path],  # Path to MODEL-READY audio (converted upstream)
        text: str,                # Transcript text to align against the audio
        **kwargs                  # Provenance pass-through (unused by FA compute)
    ) -> ForcedAlignResult:       # Word-level alignment result
        """Align transcript text to model-ready audio at word level — PURE COMPUTE.

        Stage 8 / PILLAR 1c: the cache-check + persistence bookends + per-call
        `force` moved to the generic forced-alignment adapter; this method loads
        the model, runs Qwen3, and builds the typed result. The alignment language
        comes from `self.config.language` (no per-call kwarg override — the tool
        runs its effective config; the prior unhashed `language` override was
        retired at migration)."""
        if not isinstance(audio, (str, Path)):
            raise PluginInputError(  # SG-47: typed input-validation (multi-inherits ValueError)
                f"Unsupported audio type: {type(audio)}; expected a file path (str or Path)",
                fields_invalid=["audio"],
            )
        audio_path = str(audio)

        # Pure compute: lazy-load the model, then align (no cache/persist here).
        self._load_model()

        self.report_progress(0.4, "Running forced alignment...")
        self.logger.info(f"Running alignment on {audio_path} ({len(text)} chars)")

        # SG-47 Track B wraps the inference site so CUDA OOM surfaces as
        # PluginResourceError (via cjm-torch-plugin-utils) → CR-7 reactive-retry reloads.
        language = self.config.language
        try:
            results = self._model.align(
                audio=audio_path,
                text=text,
                language=language,
            )
        except torch.cuda.OutOfMemoryError as e:
            raise cuda_oom_to_plugin_resource_error(
                e, label=f"during Qwen3 forced-alignment (text_chars={len(text)})",
            ) from e

        self.report_progress(0.8, "Processing results...")

        # Convert qwen_asr ForcedAlignResult to our standardized DTO.
        items = [
            ForcedAlignItem(text=item.text, start_time=item.start_time, end_time=item.end_time)
            for item in results[0].items
        ]
        result = ForcedAlignResult(
            items=items,
            metadata={
                "model_id": self.config.model_id,
                "language": language,
                "device": self._loaded_device,
                "dtype": self.config.dtype,
                "word_count": len(items),
            }
        )
        self.report_progress(1.0, "Alignment complete.")
        self.logger.info(f"Alignment complete: {len(items)} words")
        return result

In [ ]:
#| export
@patch
def is_available(self:Qwen3ForcedAlignerPlugin) -> bool:  # True if qwen_asr is importable
    """Check if the Qwen3 forced aligner is available."""
    return QWEN3_AVAILABLE

In [ ]:
#| export
@patch
def _apply_config(
    self:Qwen3ForcedAlignerPlugin,
    config: Optional[Any] = None  # Configuration dataclass, dict, or None
) -> None:
    """CR-4: apply config values only. Called by initialize (first-time) and the
    substrate's reconfigure delta path. Model release on a model_id/device/dtype/
    attn_implementation change is handled declaratively via RELOAD_TRIGGER ->
    _release_model (device/dtype are resolved lazily in _load_model)."""
    self.config = dict_to_config(Qwen3ForcedAlignerConfig, config or {})

In [ ]:
#| export
@patch
def _load_model(self:Qwen3ForcedAlignerPlugin):
    """Load model on first use. Heartbeat-wrapped HF Hub download + build."""
    if self._model is not None:
        return

    self.report_progress(0.0, "Loading Qwen3 Forced Aligner model...")

    device = self.config.device
    if device == "auto":
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

    dtype_map = {
        "bfloat16": torch.bfloat16,
        "float16": torch.float16,
        "float32": torch.float32,
    }
    dtype = dtype_map.get(self.config.dtype, torch.bfloat16)

    self.report_progress(0.1, f"Loading model {self.config.model_id}...")
    self.logger.info(f"Loading model {self.config.model_id} on {device} with {self.config.dtype}")

    # Heartbeat wraps the WHOLE download + build: snapshot_download_with_progress
    # streams the HF Hub download (per-file tqdm -> report_progress is sugar) and the
    # heartbeat is the stall-detector floor for the silent stretches. Qwen3ForcedAligner
    # .from_pretrained forwards **kwargs to AutoModel.from_pretrained, so cache_dir /
    # revision / local_files_only / trust_remote_code apply; cache_dir stays at the
    # HFCacheConfig default (None -> HF_HOME) so the model + the separately-loaded
    # AutoProcessor share one snapshot-populated cache.
    with self.heartbeat("loading Qwen3 model"):
        snapshot_download_with_progress(
            self.config.model_id,
            report_progress=self.report_progress,
            cache_dir=self.config.cache_dir,
            revision=self.config.revision,
            local_files_only=self.config.local_files_only,
        )
        self._model = load_pretrained_with_oom(
            Qwen3ForcedAligner,
            self.config.model_id,
            label=f"loading Qwen3 model {self.config.model_id!r}",
            dtype=dtype,
            device_map=device,
            attn_implementation=self.config.attn_implementation,
            cache_dir=self.config.cache_dir,
            revision=self.config.revision,
            local_files_only=True,
            trust_remote_code=self.config.trust_remote_code,
        )

    self._loaded_model_id = self.config.model_id
    self._loaded_device = device
    self._loaded_dtype = self.config.dtype
    self._loaded_attn = self.config.attn_implementation

    self.report_progress(0.25, "Model loaded.")
    self.logger.info("Model loaded successfully")

In [ ]:
#| export
@patch
def _release_model(self:Qwen3ForcedAlignerPlugin):
    """CR-4: release the model + free CUDA cache (cjm-torch-plugin-utils).
    RELOAD_TRIGGER target for model_id/device/dtype/attn_implementation; on_disable /
    cleanup delegate here. Idempotent (release_model no-ops when already None)."""
    release_model(self, ["_model"], device="cuda", logger=self.logger)
    self._loaded_model_id = None
    self._loaded_device = None
    self._loaded_dtype = None
    self._loaded_attn = None

In [ ]:
#| export
@patch
def prefetch(self:Qwen3ForcedAlignerPlugin) -> None:
    """CR-4 (SG-19): eagerly load the model so the first execute() doesn't pay
    the download/load cost. Idempotent via _load_model's None-guard."""
    self._load_model()

In [ ]:
#| export
@patch
def on_disable(self:Qwen3ForcedAlignerPlugin) -> None:
    """CR-2: release the GPU model when the operator disables the plugin (the
    worker stays alive); lazy reload on the next execute after re-enable."""
    self._release_model()

In [ ]:
#| export
@patch
def cleanup(self:Qwen3ForcedAlignerPlugin) -> None:
    """Clean up resources."""
    self._release_model()
    self.logger.info("Cleanup completed")

## Testing

In [ ]:
# Test basic plugin properties
plugin = Qwen3ForcedAlignerPlugin()

# NOTE: plugin.name derives from the installed distribution (importlib.metadata);
# it resolves in the worker / installed module, NOT in __main__ (where __package__
# is None), so it is asserted by the manifest generator + e2e, not here.
print(f"Plugin version: {plugin.version}")
print(f"Available: {plugin.is_available()}")
print(f"Config class: {plugin.config_class.__name__}")
assert isinstance(plugin, ToolCapability)
assert plugin.config_class.__name__ == "Qwen3ForcedAlignerConfig"

In [ ]:
# Test configuration
from dataclasses import fields

print("Config fields:")
for f in fields(Qwen3ForcedAlignerConfig):
    print(f"  {f.name}: default={f.default}, title={f.metadata.get(SCHEMA_TITLE, '')}")

Config fields:
  model_id: default=Qwen/Qwen3-ForcedAligner-0.6B, title=Model ID
  device: default=auto, title=Device
  dtype: default=bfloat16, title=Data Type
  attn_implementation: default=sdpa, title=Attention Implementation
  language: default=English, title=Language


In [ ]:
# Test JSON Schema generation
import json

plugin = Qwen3ForcedAlignerPlugin()

schema = plugin.get_config_schema()
print(f"Schema name: {schema['name']}")
print(f"Properties: {len(schema['properties'])}")
print(json.dumps(schema['properties'], indent=2))

Schema name: Qwen3ForcedAlignerConfig
Properties: 5
{
  "model_id": {
    "type": "string",
    "title": "Model ID",
    "description": "HuggingFace model identifier",
    "enum": [
      "Qwen/Qwen3-ForcedAligner-0.6B"
    ],
    "default": "Qwen/Qwen3-ForcedAligner-0.6B"
  },
  "device": {
    "type": "string",
    "title": "Device",
    "description": "Device for model inference ('auto', 'cuda:0', 'cpu')",
    "default": "auto"
  },
  "dtype": {
    "type": "string",
    "title": "Data Type",
    "description": "Model precision",
    "enum": [
      "bfloat16",
      "float16",
      "float32"
    ],
    "default": "bfloat16"
  },
  "attn_implementation": {
    "type": "string",
    "title": "Attention Implementation",
    "description": "Attention backend ('sdpa' for PyTorch native, 'flash_attention_2' if flash-attn installed)",
    "enum": [
      "sdpa",
      "flash_attention_2",
      "eager"
    ],
    "default": "sdpa"
  },
  "language": {
    "type": "string",
    "title": "

In [ ]:
# Test initialization
import logging
logging.basicConfig(level=logging.INFO)

plugin.initialize({"language": "English", "device": "auto"})
print(f"Current config: {plugin.get_current_config()}")

# Model should NOT be loaded yet (lazy loading)
assert plugin._model is None
print("Model not loaded (lazy): OK")

Initialized with model=Qwen/Qwen3-ForcedAligner-0.6B, device=auto


Current config: {'model_id': 'Qwen/Qwen3-ForcedAligner-0.6B', 'device': 'auto', 'dtype': 'bfloat16', 'attn_implementation': 'sdpa', 'language': 'English'}
Model not loaded (lazy): OK


In [ ]:
# Test config change detection
print("Re-initializing with same config (no reload)...")
plugin.initialize({"language": "English", "device": "auto"})

print("Re-initializing with different dtype (would reload if model loaded)...")
plugin.initialize({"language": "English", "device": "auto", "dtype": "float16"})
print(f"Updated dtype: {plugin.config.dtype}")

# Cleanup
plugin.cleanup()
print("Cleanup: OK")

### Execution Tests

These tests require the Qwen3 model to be downloaded and GPU available.

In [ ]:
#| eval: false
from pathlib import Path

# Test pure-compute alignment with real audio (cache/persist now live in the adapter)
plugin = Qwen3ForcedAlignerPlugin()
plugin.initialize({"language": "English"})

test_audio = Path("../test_files/short_test_audio.mp3")
test_text = Path("../test_files/short_test_audio.txt").read_text().strip()

print(f"Audio: {test_audio}")
print(f"Text: {test_text[:80]}...")

result = plugin.align(str(test_audio), text=test_text)

print(f"\nResult: {len(result.items)} words")
print(f"Metadata: {result.metadata}")
print(f"\nFirst 10 items:")
for item in result.items[:10]:
    print(f"  {item}")

plugin.cleanup()

In [ ]:
#| eval: false
# Test with longer audio
plugin = Qwen3ForcedAlignerPlugin()
plugin.initialize({"language": "English"})

test_audio = Path("../test_files/02 - 1. Laying Plans.mp3")
test_text = Path("../test_files/02 - 1. Laying Plans.txt").read_text().strip()

print(f"Audio: {test_audio}")
print(f"Text length: {len(test_text)} chars, {len(test_text.split())} words")

result = plugin.align(str(test_audio), text=test_text)

print(f"\nResult: {len(result.items)} words aligned")
print(f"First 5: {result.items[:5]}")
print(f"Last 5: {result.items[-5:]}")

plugin.cleanup()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()